In [1]:
import pandas as pd
import numpy as np


In [2]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv', dtype={"jnc": str})    


In [3]:

intervals=pd.read_csv('../data/interim/quantile_intervals_groupkfold.csv', index_col=0)

comps=pd.read_csv('../data/interim/nn_comps.csv', index_col=0)

In [9]:
def build_valuation_table(df, intervals, comps):
    base_cols = [
    "jnc", "Price_yen", "municipality", "area_m2", "age_years",
    "walk_min_nearest", "n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R",
    "building_key", "lookalike_id"
]

    interval_cols = ["lo2", "hi2", "mid"]
    if not set(interval_cols).issubset(intervals.columns):
        interval_cols = ["lo", "hi", "mid"]

    val = (
        df[base_cols]
        .join(intervals[interval_cols])
        .join(comps[["comp_est", "comp_lo", "comp_hi", "comp_n"]])
        .rename(columns={
            interval_cols[0]: "lo",
            interval_cols[1]: "hi",
            "mid": "estimate",
        })
    )

    # Convert log-space predictions to yen if necessary.
    if val[["lo", "estimate", "hi"]].median().median() < 30:
        val[["lo", "estimate", "hi"]] = np.expm1(
            val[["lo", "estimate", "hi"]]
        )

    assert len(val) == len(df)
    assert val[["estimate", "lo", "hi"]].notna().all().all()
    assert (val["lo"] <= val["estimate"]).all()
    assert (val["estimate"] <= val["hi"]).all()

    val["position"] = np.select(
        [
            val["Price_yen"] > val["hi"],
            val["Price_yen"] < val["lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["gap_pct"] = (val["Price_yen"] - val["estimate"]) / val["estimate"]

    val["comp_position"] = np.select(
        [
            val["Price_yen"] > val["comp_hi"],
            val["Price_yen"] < val["comp_lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["region"] = np.select(
        [
            val["municipality"].astype(str).str.endswith("区"),
            val["municipality"].astype(str).str.endswith("市"),
        ],
        ["ward", "city"],
        default="town_village",
    )

    val["confidence"] = np.where(
        (val["region"] == "town_village")
        | (val["area_m2"] > 80)
        | (val["comp_n"] < 5),
        "low",
        "normal",
    )

    val["label"] = np.select(
        [
            (val["position"] == "above") & (val["comp_position"] == "above"),
            (val["position"] == "above"),
            (val["position"] == "below") & (val["comp_position"] == "below"),
            (val["position"] == "below"),
        ],
        [
            "Relatively high vs model and similar listings",
            "Above the model's expected range",
            "Relatively low vs model and similar listings",
            "Below the model's expected range",
        ],
        default="Within the expected range",
    )

    return val


val = build_valuation_table(df, intervals, comps)

print("Rows:", len(val))
print("\nNaN counts:")
print(val.isna().sum())

print("\nPosition shares (%):")
print(val["position"].value_counts(normalize=True).mul(100).round(2))

print("\nLow-confidence share (%):")
print((val["confidence"].eq("low").mean() * 100).round(2))

sample_cols = (
    val.select_dtypes(exclude=["object", "string", "category"])
    .columns.difference(["jnc", "building_key"], sort=False)
)

print("\nRandom sample:")
display(val.sample(5, random_state=42)[sample_cols])


assert val["comp_n"].max() <= 10
assert (val["comp_est"] != val["estimate"]).mean() > 0.99
assert 0.78 < (val["position"] == "within").mean() < 0.82

val.to_csv("../data/interim/valuation_table.csv", index=True)

Rows: 183490

NaN counts:
jnc                    0
Price_yen              0
municipality           0
area_m2                0
age_years              0
walk_min_nearest    1665
n_rooms                0
has_L                  0
has_D                  0
has_K                  0
has_S                  0
is_1R                  0
building_key           0
lookalike_id           0
lo                     0
hi                     0
estimate               0
comp_est               0
comp_lo                0
comp_hi                0
comp_n                 0
position               0
gap_pct                0
comp_position          0
region                 0
confidence             0
label                  0
dtype: int64

Position shares (%):
position
within    79.89
above     10.14
below      9.97
Name: proportion, dtype: float64

Low-confidence share (%):
2.79

Random sample:


,Price_yen,area_m2,age_years,walk_min_nearest,n_rooms,has_L,has_D,has_K,has_S,is_1R,lookalike_id,lo,hi,estimate,comp_est,comp_lo,comp_hi,comp_n,gap_pct
59113,97000,46.31,14,7.0,1,1,1,1,0,0,59384,73106.625000,126635.843750,93406.054688,104143.036512,101847.241512,110724.469633,10.0,0.038477
119992,126000,25.84,1,9.0,1,0,0,1,0,0,37395,102299.445312,135031.609375,122719.617188,137772.511848,137772.511848,138793.048973,10.0,0.026731
104736,90000,31.88,24,3.0,1,0,1,1,0,0,44566,77092.867188,116859.304688,92193.515625,91074.806516,83654.814351,117016.067167,10.0,-0.023793
94546,69000,21.12,5,8.0,1,0,0,0,0,1,144623,73481.914062,97593.921875,82859.601562,97959.183673,83355.206847,100897.959184,10.0,-0.167266
76622,70000,35.00,46,5.0,1,0,1,1,0,0,152487,68728.695312,118102.484375,88143.187500,87943.192734,84045.524377,94424.235163,10.0,-0.205838


In [5]:
print(val["comp_n"].describe())       # real NN output: max <= 10
print((val["comp_est"] == val["estimate"]).mean())   # should be near 0, not 1
print((val["comp_n"] < 5).sum())

count    183490.000000
mean          9.999869
std           0.032348
min           2.000000
25%          10.000000
50%          10.000000
75%          10.000000
max          10.000000
Name: comp_n, dtype: float64
0.0
3


In [11]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Could not find src directory: {project_root / 'src'}")

sys.path.insert(0, str(project_root))

from src.recommender import recommend, madori_label

val = pd.read_csv("../data/interim/valuation_table.csv", index_col=0)

rooms = pd.read_csv(
    "../data/interim/listings_parsed_v7.csv",
    usecols=["Rooms"],
)["Rooms"]

labels = val.apply(madori_label, axis=1)
print((labels == rooms.values).mean())

out, msg = recommend(
    val,
    {"max_rent": 100000, "madori": ["1LDK", "2LDK"], "max_walk": 10},
)

print(len(out), msg)
display(out.head())

1.0
20 Dropped 18 low-confidence row(s). Excluded 143 listing(s) with a gap beyond 30% (likely model error). Dropped 1277 duplicate look-alike row(s). Ranked by relatively low asking rent vs similar listings, walk, age and area.
Not a guarantee of a good deal.


,Price_yen,estimate,lo,hi,position,gap_pct,walk_min_nearest,age_years,area_m2,municipality,n_rooms,score,units_in_building
146836,85000,118214.492188,100075.390625,160404.203125,below,-0.280968,2.0,5,63.34,府中市,2,0.639373,1
35660,68000,96727.453125,76513.710938,141263.046875,below,-0.296994,2.0,3,45.40,武蔵村山市,1,0.612105,1
64058,90000,125902.843750,105192.890625,154455.250000,below,-0.285163,1.0,8,41.34,足立区,1,0.609651,1
54041,82000,115075.867188,100079.773438,144127.281250,below,-0.287427,2.0,2,41.10,立川市,1,0.601912,1
12796,75000,94646.554688,79308.460938,112685.429688,below,-0.207578,1.0,0,39.55,日野市,1,0.600832,2


In [7]:
print(out[["Price_yen","estimate","gap_pct","walk_min_nearest","age_years","area_m2"]].head(10))
print((out["gap_pct"] <= -0.29).sum())

        Price_yen       estimate   gap_pct  walk_min_nearest  age_years  \
146836      85000  118214.492188 -0.280968               2.0          5   
35660       68000   96727.453125 -0.296994               2.0          3   
64058       90000  125902.843750 -0.285163               1.0          8   
54041       82000  115075.867188 -0.287427               2.0          2   
12796       75000   94646.554688 -0.207578               1.0          0   
138436      82000  107685.234375 -0.238521               1.0         15   
5682        72000   96992.382812 -0.257674               1.0         19   
54700       79000  102653.820312 -0.230423               1.0          0   
50664       72000   96329.304688 -0.252564               1.0         19   
151948      78000   95574.828125 -0.183886               1.0          4   

        area_m2  
146836    63.34  
35660     45.40  
64058     41.34  
54041     41.10  
12796     39.55  
138436    54.50  
5682      56.30  
54700     32.67  
50664     55

In [8]:
import src.recommender, inspect
print(src.recommender.__file__)
print("units_in_building" in inspect.getsource(src.recommender.recommend))

c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\src\recommender.py
True
